# ÖBB Job Scraper
Scrapes oebb.csod.com via JSON API, scores against CV, writes to SQLite.

**Run order:** Execute cells top to bottom. Sections 3 and 5 make network calls — don't re-run accidentally.

**Cookie refresh:** Session cookie in `config_private.py` expires after ~24h. If Section 5 returns 401/403, grab a fresh cookie from DevTools → Network → a detail page request → Request Headers → Cookie.

## 0. Imports & setup

In [1]:
import sys, os
import requests
from bs4 import BeautifulSoup
import re
import time
import json
import sqlite3
from datetime import date
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

sys.path.append(os.getcwd())
from config_isis import (
    CV_TEXT, HIGH_WEIGHT_TERMS, MEDIUM_WEIGHT_TERMS,
    IGNORE_TERMS, ALLOWED_LOCATIONS, MIN_SCORE, KEYWORD_BONUS_CAP
)

from config_private import OEBB_COOKIE, OEBB_BEARER

LISTING_HEADERS = {
    'Content-Type': 'application/json',
    'Accept': 'application/json',
    'Authorization': f'Bearer {OEBB_BEARER}',
    'Csod-Accept-Language': 'en-US',
    'Origin': 'https://oebb.csod.com',
    'Referer': 'https://oebb.csod/',
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36',
}

print('Imports OK')
print(f'Config loaded — {len(HIGH_WEIGHT_TERMS)} high-weight terms, {len(IGNORE_TERMS)} ignore terms')

Imports OK
Config loaded — 21 high-weight terms, 27 ignore terms


## 1. CV profile & scoring config
Loaded from `config_isis.py`. Nothing to edit here — run to verify what was imported.

In [2]:
print('=== CV TEXT (first 300 chars) ===')
print(CV_TEXT[:300])
print()
print('=== HIGH WEIGHT TERMS ===')
print(HIGH_WEIGHT_TERMS)
print()
print('=== ALLOWED LOCATIONS ===')
print(ALLOWED_LOCATIONS)

=== CV TEXT (first 300 chars) ===

Senior Analytics und Insights Führungskraft mit 10 Jahren Erfahrung im E-Commerce.
SQL Experte BigQuery Datenanalyse Business Intelligence KPI Kennzahlen Reporting.
Kundenfeedback Kundeninsights Voice of Customer VoC NPS Retourenanalyse Funnel-Analyse.
Machine Learning ML Python scikit-learn NLP Te

=== HIGH WEIGHT TERMS ===
['sql', 'datenanalyse', 'data analysis', 'analytics', 'business intelligence', 'bi', 'looker', 'product analytics', 'kundeninsights', 'customer insights', 'kpi', 'dashboard', 'reporting', 'e-commerce', 'ecommerce', 'digital', 'retouren', 'merchandising', 'digitalisierung', 'daten', 'data']

=== ALLOWED LOCATIONS ===
['berlin', 'wien', 'vienna', 'remote', 'wo du willst', 'home office', 'homeoffice']


## 2. Scraper config (ÖBB-specific)
**This is the only section to update when adjusting ÖBB scraping behaviour.**

Key difference from DB/Siemens Energy: ÖBB uses a JSON POST API for listings (no HTML parsing needed).
Location filter is applied server-side via `states: ["wien"]` in the request payload.
Description requires an authenticated detail API call using session cookie from `config_private.py`.

⚠️ **Cookie expires ~24h.** Refresh from DevTools if detail fetches return 401/403.

In [3]:
# ── Identity ──────────────────────────────────────────────────────────────────
SOURCE = 'oebb'
DB_PATH = 'data/jobs.db'

# ── API endpoints ─────────────────────────────────────────────────────────────
LISTING_API_URL = 'https://eu-cdg-hs.api.csod.com/rec-job-search/external/jobs'
DETAIL_API_URL = 'https://oebb.csod.com/Services/API/ATS/CareerSite/4/JobRequisitions/{req_id}?useMobileAd=false&cultureId=4'

# ── Pagination ────────────────────────────────────────────────────────────────
PAGE_SIZE = 25
# page_num removed — pagination handled dynamically in Section 3

# ── Request config ────────────────────────────────────────────────────────────

DETAIL_HEADERS = {
    'Accept': 'application/json; q=1.0, text/*; q=0.8, */*; q=0.1',
    'Accept-Language': 'en-US,en;q=0.9,de-DE;q=0.8,de;q=0.7,da;q=0.6,sv;q=0.5',
    'Authorization': f'Bearer {OEBB_BEARER}',
    'Cache-Control': 'no-cache',
    'Cookie': OEBB_COOKIE,
    'Csod-Accept-Language': 'en-US',
    'Referer': 'https://oebb.csod.com/ux/ats/careersite/4/home/requisition/25444?c=oebb',
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36',
}
LISTING_PAYLOAD_TEMPLATE = {
    "careerSiteId": 4,
    "careerSitePageId": 4,
    "pageNumber": 1,  # overridden per-call in fetch_listing_page()
    "pageSize": 25,
    "cultureId": 4,
    "cultureName": "de-DE",
    "searchText": "",
    "states": ["wien"],
    "cities": [],
    "countryCodes": [],
    "customFieldCheckboxKeys": [],
    "customFieldDropdowns": [],
    "customFieldRadios": [],
    "placeID": "",
    "postingsWithinDays": None,
    "radius": None,
}

# ── Timing ────────────────────────────────────────────────────────────────────
SLEEP_BETWEEN_PAGES = 1.5
SLEEP_BETWEEN_DETAILS = 1.0

# ── Company-specific keyword overrides ───────────────────────────────────────
EXTRA_IGNORE = [
    'lokführer', 'triebfahrzeug', 'gleisbau', 'fahrdienstleiter',
    'lehrling', 'lehrstelle', 'ausbildung', 'praktikum', 'werkstudent',
    'reinigung', 'gastronomie', 'koch', 'küche',
]
EXTRA_HIGH_WEIGHT = []
EXTRA_MEDIUM_WEIGHT = []

IGNORE_TERMS_FINAL = IGNORE_TERMS + EXTRA_IGNORE
HIGH_WEIGHT_FINAL = HIGH_WEIGHT_TERMS + EXTRA_HIGH_WEIGHT
MEDIUM_WEIGHT_FINAL = MEDIUM_WEIGHT_TERMS + EXTRA_MEDIUM_WEIGHT

print(f'Source: {SOURCE}')
print(f'Page size: {PAGE_SIZE} | Location filter: wien (server-side)')
print(f'Ignore terms (total): {len(IGNORE_TERMS_FINAL)}')

Source: oebb
Page size: 25 | Location filter: wien (server-side)
Ignore terms (total): 40


In [4]:
import requests

# Test the exact request the scraper is making
test_url = "https://eu-cdg-hs.api.csod.com/rec-job-search/external/jobs"
test_payload = {
    "countryCodes": ["at"],
    "stateCodes": ["wien"],
    "pageSize": 5,
    "pageNumber": 1
}

print("Bearer (first 30 chars):", OEBB_BEARER[:30])
print("Bearer (last 10 chars):", OEBB_BEARER[-10:])
print()

resp = requests.post(test_url, headers=LISTING_HEADERS, json=test_payload, timeout=15)
print("Status:", resp.status_code)
print("Response:", resp.text[:500])

Bearer (first 30 chars): eyJhbGciOiJIUzUxMiIsInR5cCI6Ik
Bearer (last 10 chars): LO6Ze0_rtQ

Status: 200
Response: {"status":"Success","timestamp":"2026-05-15T10:42:52.2492223Z","data":{"totalCount":0,"requisitions":[],"filters":[],"customFieldFilters":[]}}


## 3. Scrape listing pages
POSTs to the ÖBB jobs API, paginates until no results remain.
Location filter (Wien) is applied server-side — no post-fetch filtering needed.

⚠️ **Network calls happen here.** Don't re-run unless you want to re-scrape.

In [5]:
def fetch_listing_page(page_number):
    """POST to ÖBB jobs API for one page. Returns list of raw job dicts."""
    payload = {**LISTING_PAYLOAD_TEMPLATE, 'pageNumber': page_number}
    try:
        resp = requests.post(LISTING_API_URL, headers=LISTING_HEADERS, json=payload, timeout=15)
        resp.raise_for_status()
    except requests.RequestException as e:
        print(f'  ⚠️  Page {page_number} failed: {e}')
        return [], 0

    data = resp.json()
    total = data.get('data', {}).get('totalCount', 0)
    requisitions = data.get('data', {}).get('requisitions', [])

    jobs = []
    for r in requisitions:
        locations = r.get('locations', [])
        location = ', '.join(
            filter(None, [locations[0].get('city', ''), locations[0].get('state', ''), locations[0].get('country', '')])
        ) if locations else ''

        jobs.append({
            'source': SOURCE,
            'source_job_id': str(r.get('requisitionId', '')),
            'title': r.get('displayJobTitle', ''),
            'company': 'ÖBB',
            'location': location,
            'start_date': r.get('postingEffectiveDate', ''),
            'link': f"https://oebb.csod.com/ux/ats/careersite/4/home/requisition/{r.get('requisitionId')}?c=oebb",
            'description': '',
        })

    return jobs, total


# ── Run scrape ────────────────────────────────────────────────────────────────
raw_jobs = []
page = 1
total_count = None

print('Scraping ÖBB jobs (Wien, paginating until exhausted)...')

while True:
    jobs, total = fetch_listing_page(page)
    if total_count is None:
        total_count = total
        print(f'Total Wien jobs reported by API: {total_count}')
    if not jobs:
        print(f'  Page {page}: no results, stopping')
        break
    raw_jobs.extend(jobs)
    print(f'  Page {page}: {len(jobs)} jobs (running total: {len(raw_jobs)})')
    if len(raw_jobs) >= total_count:
        break
    page += 1
    time.sleep(SLEEP_BETWEEN_PAGES)

# Deduplicate by source_job_id
seen = set()
unique_jobs = []
for j in raw_jobs:
    if j['source_job_id'] not in seen:
        seen.add(j['source_job_id'])
        unique_jobs.append(j)

print(f'\n✅ {len(unique_jobs)} unique Wien jobs scraped')

Scraping ÖBB jobs (Wien, paginating until exhausted)...
Total Wien jobs reported by API: 120
  Page 1: 25 jobs (running total: 25)
  Page 2: 25 jobs (running total: 50)
  Page 3: 25 jobs (running total: 75)
  Page 4: 25 jobs (running total: 100)
  Page 5: 20 jobs (running total: 120)

✅ 120 unique Wien jobs scraped


In [6]:
# Inspect raw results
df_raw = pd.DataFrame(unique_jobs)
print(f'Shape: {df_raw.shape}')
df_raw[['title', 'source_job_id', 'location']].head(10)

Shape: (120, 8)


,title,source_job_id,location
0,Junior Verkehrsplaner:in für Baustellen im Fer...,25398,"Wien-Favoriten, Wien, AT"
1,Produktionsplaner:in,25367,"Wien-Favoriten, Wien, AT"
2,Data Engineer (m/w/x),25444,"Wien-Favoriten, Wien, AT"
3,Senior Service Manager:in Data Integration,25440,"Wien-Leopoldstadt, Wien, AT"
4,People Manager:in Data & AI,25439,"Wien-Leopoldstadt, Wien, AT"
5,Zentrale:r Senior Elektrotechniker:in (Proofma...,25217,"Wien-Leopoldstadt, Wien, AT"
6,Senior Data Engineer (m/w/x),25418,"Wien-Leopoldstadt, Wien, AT"
7,Geschäftsführung (m/w/x) für Rail Cargo Logist...,25438,"Wien-Favoriten, Wien, AT"
8,Regionale:r Bautechniker:in für Hochbauanlagen,23259,"Wien-Leopoldstadt, Wien, AT"
9,Küchenhilfe (m/w/x),25437,"Wien-Favoriten, Wien, AT"


## 4. Filter
Drops irrelevant jobs by title before fetching descriptions.

Check the dropped list — if real jobs are being filtered, update EXTRA_IGNORE in Section 2 and re-run this cell only (no re-scraping needed).

In [7]:
def is_irrelevant(title):
    title_lower = title.lower()
    return any(term in title_lower for term in IGNORE_TERMS_FINAL)

before = len(unique_jobs)
filtered_jobs = [j for j in unique_jobs if not is_irrelevant(j['title'])]
dropped = [j for j in unique_jobs if is_irrelevant(j['title'])]

print(f'Before filter: {before}')
print(f'After filter:  {len(filtered_jobs)}')
print(f'Dropped:       {len(dropped)}')
print()
print('--- Dropped titles (sanity check) ---')
for j in dropped:
    print(f"  {j['title']}")

Before filter: 120
After filter:  110
Dropped:       10

--- Dropped titles (sanity check) ---
  Regionale:r Bautechniker:in für Hochbauanlagen
  Küchenhilfe (m/w/x)
  Junior Produktmanager:in Reinigung - Schwerpunkt Beschaffung & Verträge (mind. 30 Std.)
  Köchin / Koch
  Projekt Leistungssportler:innen werden Fahrdienstleiter:innen (Region Ost)
  Facility Monteur:in für ÖBB-Objekte
  Fallweise Aushilfe (m/w/x) Geringfügig  Service und Küche für interne Veranstaltungen in Wien
  Ausgebildete:r Lokführer:in Wien
  Lokführer:in Quereinstieg, Wien
  Internationaler Ganzzug: Ausgebildete:r Lokführer:in


## 5. Fetch descriptions
Calls the authenticated ÖBB detail API for each job. Strips HTML tags from the `ad` field.

⚠️ **Requires valid session cookie in `config_private.py`.** If you get 401/403 errors, refresh the cookie.
⚠️ **Slow cell.** Checkpoint JSON saved after completion — reload from it if kernel restarts.

In [8]:
def fetch_description(job):
    """
    Calls ÖBB detail API. Extracts full JD from the 'ad' HTML field.
    Returns clean plain text.
    """
    req_id = job['source_job_id']
    url = DETAIL_API_URL.format(req_id=req_id)

    try:
        resp = requests.get(url, headers=DETAIL_HEADERS, timeout=15)
        if resp.status_code in (401, 403):
            print(f"    ⚠️  Auth failed ({resp.status_code}) — refresh OEBB_COOKIE in config_private.py")
            return job['title']
        resp.raise_for_status()
    except requests.RequestException as e:
        print(f"    ⚠️  Failed: {job['title'][:40]}: {e}")
        return job['title']

    data = resp.json()

    # Navigate to the job ad HTML — structure: data.data[0].items[0].fields.ad
    try:
        ad_html = data['data'][0]['items'][0]['fields']['ad']
    except (KeyError, IndexError, TypeError):
        # Fallback: try description field
        try:
            ad_html = data['data'][0]['items'][0]['fields']['description']
        except (KeyError, IndexError, TypeError):
            return job['title']

    # Strip HTML tags
    clean = BeautifulSoup(ad_html, 'html.parser').get_text(separator=' ', strip=True)
    clean = re.sub(r'\s+', ' ', clean).strip()

    return clean if clean else job['title']


enriched_jobs = []
print(f'Fetching descriptions for {len(filtered_jobs)} jobs...')

for i, job in enumerate(filtered_jobs, 1):
    print(f"  [{i}/{len(filtered_jobs)}] {job['title'][:50]}...", end=' ')
    desc = fetch_description(job)
    job['description'] = desc
    enriched_jobs.append(job)
    print(f'{len(desc.split())} words')
    time.sleep(SLEEP_BETWEEN_DETAILS)

checkpoint_path = f'data/{SOURCE}_checkpoint.json'
with open(checkpoint_path, 'w', encoding='utf-8') as f:
    json.dump(enriched_jobs, f, ensure_ascii=False, indent=2)
print(f'\n✅ Descriptions fetched. Checkpoint saved to {checkpoint_path}')

Fetching descriptions for 110 jobs...
  [1/110] Junior Verkehrsplaner:in für Baustellen im Fernver... 697 words
  [2/110] Produktionsplaner:in... 676 words
  [3/110] Data Engineer (m/w/x)... 502 words
  [4/110] Senior Service Manager:in Data Integration... 784 words
  [5/110] People Manager:in Data & AI... 824 words
  [6/110] Zentrale:r Senior Elektrotechniker:in (Proofmanage... 705 words
  [7/110] Senior Data Engineer (m/w/x)... 756 words
  [8/110] Geschäftsführung (m/w/x) für Rail Cargo Logistics ... 680 words
  [9/110] Strategische:r Einkäufer:in Anlagen & Betriebsmitt... 756 words
  [10/110] Jurist:in mit Schwerpunkt Risikomanagement... 596 words
  [11/110] Teamassistent:in der Rechtsabteilung – Konzernrech... 640 words
  [12/110] Spezialist:in Engineering RAM-LCC... 664 words
  [13/110] Junior Spezialist:in Engineering: Technisches Doku... 677 words
  [14/110] Junior Spezialist:in Engineering: Konfigurationsma... 693 words
  [15/110] Spezialist:in mit betriebswirtschaftlicher Komp

In [ ]:
# Reload from checkpoint if kernel restarted after Section 5:
# with open(f'data/{SOURCE}_checkpoint.json', encoding='utf-8') as f:
#     enriched_jobs = json.load(f)
# print(f'Loaded {len(enriched_jobs)} jobs from checkpoint')

## 6. Location filter
Location is filtered server-side in Section 3 (states: ["wien"]), so this is a safety check only.

In [9]:
def is_allowed_location(location):
    loc_lower = location.lower()
    return any(allowed in loc_lower for allowed in ALLOWED_LOCATIONS)

before = len(enriched_jobs)
location_filtered = [j for j in enriched_jobs if is_allowed_location(j['location'])]
dropped_location = [j for j in enriched_jobs if not is_allowed_location(j['location'])]

print(f'Before: {before}  |  After: {len(location_filtered)}  |  Dropped: {len(dropped_location)}')
if dropped_location:
    for j in dropped_location:
        print(f"  {j['title'][:50]} | {j['location']}")
else:
    print('No jobs dropped — location was pre-filtered server-side (expected)')

Before: 110  |  After: 106  |  Dropped: 4
  Senior Spezialist:in Prüftechnologien + Spurkranzs | St. Pölten, Niederösterreich, AT
  Zulassungsingenieur:in Schienenfahrzeuge | St. Pölten, Niederösterreich, AT
  Senior IT-Business Analyst:in | St. Pölten, Niederösterreich, AT
  Ferialpraktikant:in für Bahnbistros | St. Pölten, Niederösterreich, AT


## 7. Score
TF-IDF cosine similarity + keyword bonus. Review top 20 before writing to DB.

In [10]:
def keyword_bonus(text):
    text_lower = text.lower()
    bonus = 0
    for term in HIGH_WEIGHT_FINAL:
        if term in text_lower:
            bonus += 8
    for term in MEDIUM_WEIGHT_FINAL:
        if term in text_lower:
            bonus += 3
    return min(bonus, KEYWORD_BONUS_CAP)


def score_jobs(jobs):
    if not jobs:
        print('No jobs to score.')
        return jobs
    descriptions = [j['description'] for j in jobs]
    corpus = [CV_TEXT] + descriptions
    vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=5000)
    tfidf_matrix = vectorizer.fit_transform(corpus)
    similarities = cosine_similarity(tfidf_matrix[0], tfidf_matrix[1:])[0]
    for i, job in enumerate(jobs):
        base = round(float(similarities[i]) * 100, 1)
        bonus = keyword_bonus(job['description'])
        job['score'] = min(round(base + bonus, 1), 100)
    return sorted(jobs, key=lambda x: x['score'], reverse=True)


scored_jobs = score_jobs(location_filtered)
print(f'✅ Scored {len(scored_jobs)} jobs')

✅ Scored 106 jobs


In [11]:
df_scored = pd.DataFrame(scored_jobs)[['score', 'title', 'location', 'link']]
df_scored.head(20)

,score,title,location,link
0,46.6,Senior Data Engineer (m/w/x),"Wien-Leopoldstadt, Wien, AT",https://oebb.csod.com/ux/ats/careersite/4/home...
1,44.2,People Manager:in Data & AI,"Wien-Leopoldstadt, Wien, AT",https://oebb.csod.com/ux/ats/careersite/4/home...
2,43.6,Senior Service Manager:in Data Integration,"Wien-Leopoldstadt, Wien, AT",https://oebb.csod.com/ux/ats/careersite/4/home...
3,43.4,Product Manager:in ServiceNow,"Wien-Leopoldstadt, Wien, AT",https://oebb.csod.com/ux/ats/careersite/4/home...
4,43.1,Platform Administrator:in/ Core Product Owner:...,"Wien-Leopoldstadt, Wien, AT",https://oebb.csod.com/ux/ats/careersite/4/home...
5,41.2,Senior Data Architekt:in - Master Data Management,"Wien-Leopoldstadt, Wien, AT",https://oebb.csod.com/ux/ats/careersite/4/home...
6,39.0,Product Manager:in SuccessFactors & HCM,"Wien-Leopoldstadt, Wien, AT",https://oebb.csod.com/ux/ats/careersite/4/home...
7,38.0,Senior IT Business Analyst:in,"Wien-Favoriten, Wien, AT",https://oebb.csod.com/ux/ats/careersite/4/home...
8,37.9,Data Engineer (m/w/x),"Wien-Favoriten, Wien, AT",https://oebb.csod.com/ux/ats/careersite/4/home...
9,37.1,Senior Change Manager:in,"Wien-Leopoldstadt, Wien, AT",https://oebb.csod.com/ux/ats/careersite/4/home...


## 8. Write to SQLite
Upserts into `jobs` table. Safe to re-run — won't create duplicates.

⚠️ **Only run when happy with scores above.**

Requires `jobs` table to exist — run `python setup_db.py` once from repo root if not already done.

In [12]:
def write_to_db(jobs, db_path, min_score):
    today = str(date.today())
    to_write = [j for j in jobs if j['score'] >= min_score]
    skipped = len(jobs) - len(to_write)
    conn = sqlite3.connect(db_path)
    try:
        conn.executemany("""
            INSERT OR REPLACE INTO jobs
                (run_date, source, source_job_id, company, title, location,
                 start_date, link, score, description)
            VALUES
                (:run_date, :source, :source_job_id, :company, :title, :location,
                 :start_date, :link, :score, :description)
        """, [{**j, 'run_date': today} for j in to_write])
        conn.commit()
        print(f'✅ Wrote {len(to_write)} jobs to {db_path}')
        print(f'   Skipped {skipped} jobs below MIN_SCORE ({min_score})')
    except Exception as e:
        conn.rollback()
        print(f'❌ DB write failed: {e}')
        raise
    finally:
        conn.close()

write_to_db(scored_jobs, DB_PATH, MIN_SCORE)

✅ Wrote 82 jobs to data/jobs.db
   Skipped 24 jobs below MIN_SCORE (15)


In [13]:
# Verify
conn = sqlite3.connect(DB_PATH)
df_db = pd.read_sql(
    f"SELECT run_date, source, title, location, score FROM jobs WHERE source = '{SOURCE}' ORDER BY score DESC",
    conn
)
conn.close()
print(f'{len(df_db)} rows in DB for source={SOURCE}')
df_db.head(10)

82 rows in DB for source=oebb


,run_date,source,title,location,score
0,2026-05-15,oebb,Senior Data Engineer (m/w/x),"Wien-Leopoldstadt, Wien, AT",46.6
1,2026-05-15,oebb,People Manager:in Data & AI,"Wien-Leopoldstadt, Wien, AT",44.2
2,2026-05-15,oebb,Senior Service Manager:in Data Integration,"Wien-Leopoldstadt, Wien, AT",43.6
3,2026-05-15,oebb,Product Manager:in ServiceNow,"Wien-Leopoldstadt, Wien, AT",43.4
4,2026-05-15,oebb,Platform Administrator:in/ Core Product Owner:...,"Wien-Leopoldstadt, Wien, AT",43.1
5,2026-05-15,oebb,Senior Data Architekt:in - Master Data Management,"Wien-Leopoldstadt, Wien, AT",41.2
6,2026-05-15,oebb,Product Manager:in SuccessFactors & HCM,"Wien-Leopoldstadt, Wien, AT",39.0
7,2026-05-15,oebb,Senior IT Business Analyst:in,"Wien-Favoriten, Wien, AT",38.0
8,2026-05-15,oebb,Data Engineer (m/w/x),"Wien-Favoriten, Wien, AT",37.9
9,2026-05-15,oebb,Senior Change Manager:in,"Wien-Leopoldstadt, Wien, AT",37.1
